# Python essentials for analysts

*Types, collections, functions, and how to recover when code breaks*

In [Chapter 1](ch-01-analytics-and-python.qmd) we introduced the seven libraries this book uses. All seven are Python libraries, and using them requires a working knowledge of the language itself. In this chapter we learn the slice of Python an analyst uses: enough to read any line of code in this book, write your own, and recover when something breaks. If you have programmed before, skim the first sections and slow down at the tracebacks and conventions. If you do not have much programming exposure, read every section in order. The only requirement is that you run every cell yourself. Reading past them does not build the skill.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-02-python-essentials.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-02-python-essentials.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. This chapter uses only Python
# itself; on Google Colab the cell fetches the course materials so the
# same notebook works there too.
import sys
if "google.colab" in sys.modules:
    !git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Values, types, and names

We begin with the most basic operation in analysis: arithmetic on named quantities. As an example, Prairie Wholesale purchases a case of Heavy-Duty Floor Cleaner for \$18.40 and sells it for \$27.60. The profit on a case, and the margin as a fraction of the selling price, are two lines of Python:

In [ ]:
unit_cost = 18.40          # what Prairie Wholesale pays per case
unit_price = 27.60         # what the customer pays per case

profit = unit_price - unit_cost
margin = profit / unit_price
margin

Three things happened. We created **variables** (named values) with `=`. We performed arithmetic operations on them using the standard arithmetic operators. Finally, the notebook printed the value of the cell's last expression (the variable `margin`) automatically. You can use this convenent method to quickly view the value of a variable or expression in a notebook. For any other output, we use `print`.

Each value has a **type**. Four types matter in this book: `int` for whole numbers, `float` for numbers with a decimal point, `str` for text, and `bool` for the two truth values `True` and `False`.

In [ ]:
type(unit_cost), type(3), type("PW-0042"), type(unit_price > unit_cost)

The parenthesized result is a **tuple**, an ordered collection of values. Python returns one of these any time we request multiple things at once. We will see this again shortly.

Text values are **strings**. One string skill recurs throughout this book: the **f-string**, which constructs text with computed values inserted. With the `f` prefix, the braces are placeholders. The specification after the colon is the format of the inserted value. In the cell below, `.1%` formats the margin as a percentage with one decimal place, and `.2f` formats the price as a number with two decimal places:

In [ ]:
sku = "PW-0042"
print(f"{sku}: margin is {margin:.1%} on a price of ${unit_price:.2f}")

Numbers stored as text cause errors. `"27.60"` is a `str` that looks like a number. Arithmetic on it either fails or produces an unexpected result: for example, `"27.60" * 2` is the text repeated twice. Type conversion is explicit. `float("27.60")` returns the number, and `str(27.6)` returns the text. In [Chapter 3](ch-03-pandas.qmd), the first check we perform after loading real data files is that each column has the expected type.

## Collections: lists and dictionaries

So far, each variable has held a single value: one cost, one price, one margin. Business data rarely comes as single values. A morning of sales, for example, produces one total per order, and an analysis must handle all of the totals together. For this, we need containers. A **list** is a sequence of values, written with square brackets. Each value in a list has a fixed position, first to last:

In [ ]:
# Order totals (in dollars) from one Tuesday morning at Prairie Wholesale
totals = [151.20, 89.75, 402.10, 63.00, 217.45, 98.30]

len(totals), sum(totals), max(totals)

`len`, `sum`, and `max` are **functions**. A function takes inputs in parentheses and returns a value. Python itself has a small set of built-in functions. Every other function in this book comes from a **library**: a published collection of functions and related tools that other people wrote, so that we do not have to build everything from scratch. We bring a library into a notebook with an `import` statement that names it. You can therefore always see where a function comes from.

Individual elements are accessed by **index**, starting from zero. Stretches of a list are accessed by **slicing**. Zero-based counting requires an adjustment:

In [ ]:
first = totals[0]        # first element (index 0, not 1)
last = totals[-1]        # negative indices count from the end
morning_rush = totals[0:3]   # a slice: elements 0, 1, 2 (the stop index is excluded)
first, last, morning_rush

A **dictionary** (`dict`) maps keys to values. Analysts use this structure for common lookups: for example, SKU to price, state to region, or customer to segment. A dictionary is defined with curly braces and queried with square brackets:

In [ ]:
price_list = {
    "PW-0042": 27.60,   # floor cleaner
    "PW-0007": 34.10,   # deli containers
    "PW-0101": 12.95,   # copy paper
}

price_list["PW-0007"]

Querying a key that is not in the dictionary raises an error. That behavior is usually desirable, because a typo in a SKU should fail visibly. Sometimes a missing key is expected and has a sensible fallback. Then `price_list.get("PW-9999", 0.0)` returns the fallback without raising an error; here the fallback is the second argument, a price of 0.0.

Lists and dictionaries can nest inside one another. One nested structure is worth previewing here, before we formalize it in [Chapter 3](ch-03-pandas.qmd): a table is naturally a list of dictionaries, one dictionary per row.

In [ ]:
order_lines = [
    {"sku": "PW-0042", "quantity": 3, "unit_price": 27.60},
    {"sku": "PW-0007", "quantity": 10, "unit_price": 34.10},
    {"sku": "PW-0101", "quantity": 2, "unit_price": 12.95},
]

order_lines[1]["quantity"]   # return quantity from 2nd row

In [Chapter 3](ch-03-pandas.qmd) we adopt pandas, whose table type, the DataFrame, is far more capable than this list of dictionaries. Conceptually, though, a DataFrame is the same structure: a collection of rows with named columns. If you keep this picture in mind, pandas code is easier to read.

## Decisions and repetition

Much of a program's usefulness comes from applying a decision many times. In Python, the decision takes the form of an `if` statement, and the repetition takes the form of a `for` loop. Suppose orders over \$200 are marked for a supervisor's review:

In [ ]:
flagged = []                        # start with an empty list and fill it

for total in totals:                # 'total' takes each value in turn
    if total > 200:
        flagged.append(total)       # .append adds one element to the end

print(f"{len(flagged)} of {len(totals)} orders flagged: {flagged}")

Two mechanical rules govern this. First, a colon terminates the `for` and `if` lines. Second, **indentation is the syntax**: the indented lines are the body of the loop, and the loop ends where the indentation ends. Python has no braces and no `end` keyword, so the visual structure is also the syntactic structure. This is one reason well-formatted Python is pleasant to read.

The loop-and-append pattern above is so common that Python has a one-line form for the pattern: the **comprehension**. Both cells, above and below, calculate the same list. With some practice, the second is easier to read, because the whole operation is on one line:

In [ ]:
flagged = [total for total in totals if total > 200]
flagged

One caveat is worth stating before [Chapter 3](ch-03-pandas.qmd): with tables, we will almost never write loops ourselves. Libraries such as pandas are designed for large amounts of data, and they provide **vectorized** operations: methods that operate on an entire column at once. In pandas, "flag every order over 200" is a single whole-column operation, and that form is both faster and clearer than a loop we write ourselves. Loops remain the right tool for repeating an *analysis*: for example, one chart per region, or one model per segment. Comprehensions are common in idiomatic Python, and for now you need both forms.

## Functions: naming a computation

Often, the same series of operations must be performed in more than one place. Writing the same code again each time is wasteful and error-prone. We can define the code once, as a **function** with a name and a set of inputs, and call it wherever we need it. You can think of a function as a mapping from inputs to outputs, with its code as the operations that produce the outputs from the inputs. The `def` statement creates one:

In [ ]:
def line_total(quantity, unit_price, discount_pct=0.0):
    """Total for one order line after its discount."""
    return quantity * unit_price * (1 - discount_pct)

line_total(3, 27.60), line_total(3, 27.60, discount_pct=0.10)

The definition has these pieces, in order. The function's name and its input **parameters** are declared on the `def` line. The `=0.0` in `discount_pct=0.0` is a **default** value, so callers with no discount can leave that parameter out. The string under the `def` line is the **docstring**: one sentence explaining what the function does. `return` returns the result to the caller. Note that in the second call, the argument is passed by name: `discount_pct=0.10`. With named arguments, the call is self-documenting. Named arguments are used extensively in this course's libraries; `test_size=0.3` and `random_state=0` appear throughout Part IV.

Functions matter to analysts for a second reason as well: a named function is a **checkable unit**. Call `line_total` with inputs whose correct answer you can compute by hand. Three cases at \$27.60 with 10% off should be \$74.52. The check ensures the code conforms to the business rule. This is a small-scale version of the evaluation discipline we apply throughout the book.

In [ ]:
assert round(line_total(3, 27.60, discount_pct=0.10), 2) == 74.52   # no output if true, error if false

Computers represent decimals in binary, and the result of `3 * 27.60 * 0.9` is `74.52000000000001`. A direct comparison of unrounded money values is therefore unreliable. In the assertion above, we round to the cent before the comparison.

## Reading a traceback

Code fails routinely. In such cases, an informative failure report is generally generated. The report can help you understand and fix the failure, so make it a habit to read the report before changing anything. Python reports failures as a **traceback**. Read it from the bottom up: the last line states the error that was raised, and the lines above it give the location in the code where the error occurred.

In [ ]:
price_list["PW-9999"]    # a SKU that is not in the dictionary

The last line is `KeyError: 'PW-9999'`. We queried a dictionary for a key it does not have. Three failure types account for most errors in this course. `NameError` means you used a name that does not exist; the most common causes are a typo or a cell you forgot to run. `TypeError` means the value is of the wrong type for the operation; the classic case is arithmetic on text. `KeyError` and `IndexError` report the same kind of failure: a lookup that found nothing. All three specify exactly what went wrong, once you learn to read the bottom line first.

In [ ]:
"27.60" + 1.5    # text plus number will give you a TypeError

Notebooks maintain state between cells, and cells can be run in any order. It is therefore possible to reference a variable before running the cell that initializes the variable, resulting in a `NameError` for a variable you believe you defined. A notebook that produces correct results by executing cells only in some hidden order is not reproducible. We therefore recommend one habit: restart the kernel, run all cells, and confirm the results. Jupyter, Google Colab, and VS Code all provide commands for this. The restart clears the accumulated state, so the run confirms that the notebook works from top to bottom.

## Conventions that make code readable

Python, like all languages, has some conventions, or a common style. Code that follows the common style is readable to every other Python reader, including you, weeks or months later. In this course we hold you to the following subset of these conventions.

- **Variable and function names are lowercase with underscores** (`unit_price`, `line_total`), and they state what the value *is*. `revenue_by_region` is clearer than `df2`. Single letters are acceptable only for trivial loop variables.
- **We comment to give the reason behind the code, or to clarify a non-obvious line.** Comments can help colleagues or even yourself understand the reasoning behind your code. A comment that repeats the code (`x = x + 1  # add one to x`) is noise. A comment that gives the reason behind a line (`total > 200  # billing policy threshold`) is documentation.
- **Notebooks are for analysis, scripts are for programs.** All code in this course is in notebooks, where text, code, and output live together. Code that a company runs on a schedule is written as a `.py` script. A notebook that runs cleanly from top to bottom is straightforward to convert into such a script when the analysis becomes a scheduled process.
- **Hard-coded constants get names.** A value like the `200` threshold from the previous example is often a parameter of the problem: a quantity set by a policy or an assumption, subject to change. Such a value is defined once, as `REVIEW_THRESHOLD = 200`, near the top of the notebook; programmers call an unexplained numeric literal a "magic number". When the policy changes, we change one line.

## Working with AI assistants {#sec-ch2-ai}

AI assistants can generate most of the code in this chapter, and they can help you understand it. We presume you will use them. When you accept generated code, however, you are responsible for it, just as a manager who signs a report drafted by an assistant is accountable for its contents. Understanding and verifying that code are therefore essential. The verification can take the same form as the `assert` from the functions section: a call on inputs whose correct answer you know independently.

The course has three rules for assistants. Appendix D lists them in full.

1. **Where an assignment permits assistants, record the use.** State which tool you used and for which portions; one line in the notebook suffices. Unrecorded use is an academic-integrity infraction.
2. **Verify anything you take.** Call the generated function with inputs whose correct answer you can compute by hand. Read every line, and remove what you cannot explain. "The model wrote it" is not an acceptable answer in a code review or your deliverables.
3. **Do the reading and labs yourself.** On the exams, you calculate numbers from a dataset only you possess. The quizzes are closed-book concept checks. Only your understanding transfers to those settings. You develop that understanding when you use an assistant as a tutor: for example, when you ask it to explain what a traceback means. When the assistant completes the work in your place, you develop no understanding of your own. The same logic applies beyond the course: if an assistant does all of your work, you will find it difficult to justify your value to an employer.

> **Don't outsource this**
>
> In some labs, you *interpret* a result: for example, why the median is so far below the mean, or what a coefficient means for pricing. Your value as an analyst rests on two skills: identifying what needs to be done, and interpreting the results once the work is done. If you delegate these skills, you will be unable to explain your own results and recommendations, and an analyst who cannot explain their recommendations quickly loses credibility. Generated code is allowed where the assignment allows it. The written interpretation must be yours.

## Checking that you are ready for [Chapter 3](ch-03-pandas.qmd)

You are ready for [Chapter 3](ch-03-pandas.qmd) if three things are true. You can read a short block of code, like the flagging loop above, and know what it will print before running it. You can read a traceback down to the failing line. And you understand why we write `assert line_total(...) == ...` after defining a function. The build lab below covers exactly these three abilities.

## Exercises



### Build lab

Write a function `order_summary(lines)`. It takes a list of order-line dictionaries like `order_lines` above (keys `sku`, `quantity`, `unit_price`, and sometimes `discount_pct`). It returns a dictionary with three keys: `n_lines` (how many lines), `total` (the order total after any line discounts), and `biggest_sku` (the SKU of the line with the largest post-discount value). Reuse `line_total` inside it. Then demonstrate it on an order you invent with at least four lines, one of which carries a discount.

### Evaluate lab

Before running `order_summary` on your invented order, compute the three answers by hand; a calculator is fine. Write them in a markdown cell. Then run the function and `assert` that each result matches your hand computation. If an assertion fails, find the bug. State in one sentence whether the bug was in the code or in the hand computation. Both happen, which is the point of checking.